# Tabela `top3_customer_quarter` (gold)

Calcula, por ano e trimestre, os 3 clientes que mais gastaram, com a soma
dos pagamentos de cada um naquele trimestre. Fontes: `fact_payment` e
`dim_customer` (schema `silver`). A escrita é idempotente
(`createOrReplace`), então reexecuções recriam a tabela sem duplicar dados.

In [ ]:
SPARK_APP_NAME = "cria-top3-customer-quarter-gold"

from nucleo import cria_spark_session

spark = cria_spark_session(SPARK_APP_NAME)
spark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABELA_PAGAMENTO = "nessie.silver.fact_payment"
TABELA_CLIENTE = "nessie.silver.dim_customer"
TABELA_DESTINO = "nessie.gold.top3_customer_quarter"

janela = Window.partitionBy("year", "quarter").orderBy(F.desc("total_spent"))

df = (
    spark.table(TABELA_PAGAMENTO)
    .withColumn("year", F.year("payment_date"))
    .withColumn("quarter", F.quarter("payment_date"))
    .groupBy("customer_id", "year", "quarter")
    .agg(F.sum("amount").alias("total_spent"))
    .join(
        spark.table(TABELA_CLIENTE).select(
            "customer_id",
            F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")).alias("customer_name"),
        ),
        "customer_id",
    )
    # shortcut: empates no 3o lugar decididos arbitrariamente; dense_rank se o KPI exigir todos
    .withColumn("ranking", F.row_number().over(janela))
    .filter(F.col("ranking") <= 3)
    .select("year", "quarter", "ranking", "customer_id", "customer_name", "total_spent")
)
df.writeTo(TABELA_DESTINO).createOrReplace()

In [ ]:
n = spark.table(TABELA_DESTINO).count()
print(f"{TABELA_DESTINO}: {n} linhas")